In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
import math

In [40]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

In [41]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [42]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [44]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
lead_source,4852,5,organic_search,1251,NaN,NaN,NaN,NaN,NaN,NaN,NaN
industry,4760,6,technology,1173,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employment_status,4806,4,employed,2633,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,4792,5,north_america,1371,NaN,NaN,NaN,NaN,NaN,NaN,NaN
annual_income,4631.0,NaN,NaN,NaN,54533.212049,21245.66171,12000.0,40458.5,55051.0,71252.5,109886.0
number_of_courses_viewed,5000.0,NaN,NaN,NaN,2.0894,1.188987,0.0,1.0,2.0,3.0,6.0
interaction_count,5000.0,NaN,NaN,NaN,4.4726,2.435906,0.0,3.0,4.0,6.0,13.0
lead_score,4965.0,NaN,NaN,NaN,0.494767,0.199431,0.01,0.36,0.49,0.63,0.99
converted,5000.0,NaN,NaN,NaN,0.5766,0.494147,0.0,0.0,1.0,1.0,1.0


In [45]:
df["interaction_count"].nunique()

14

In [46]:
categorical_features = ["lead_source", "industry", "employment_status", "location"]
numerical_features = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score"]

In [47]:
df[categorical_features] = df[categorical_features].fillna('NA')

In [48]:
df[[col for col in df.columns if col not in categorical_features]] = df[[col for col in df.columns if col not in categorical_features]].fillna(0)

In [49]:
df


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NA,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [50]:
df["industry"].mode()

0    technology
Name: industry, dtype: str

In [51]:
numerical_features

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [52]:
(df[numerical_features].corr().values - np.eye(4)).max()

np.float64(0.9157457980418697)

In [53]:
df[numerical_features].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [54]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [55]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [56]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [57]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [58]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [59]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [60]:
mi = df_full_train[categorical_features].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.034761
employment_status    0.020313
industry             0.002898
location             0.001050
dtype: float64

In [61]:
dv = DictVectorizer(sparse = False)
train_dict = df_train[categorical_features + numerical_features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

In [62]:
X_train.shape

(3000, 28)

In [63]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [64]:
val_dict = df_val[categorical_features + numerical_features].to_dict(orient="records")
X_val = dv.transform(val_dict)

y_pred = model.predict(X_val)

In [65]:
round(float((y_pred == y_val).mean()), 2)  

0.65

In [66]:
(y_pred == y_val).mean()

np.float64(0.645)

In [67]:
all_features = categorical_features + numerical_features
baseline_accuracy = (y_pred == y_val).mean()

feature_results = []

for feature in all_features:
    selected_features = [f for f in all_features if f != feature]

    dv_new = DictVectorizer(sparse=False)

    train_dict_new = df_train[selected_features].to_dict(orient="records")
    X_train_new = dv_new.fit_transform(train_dict_new)

    model_new = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
    model_new.fit(X_train_new, y_train)

    val_dict_new = df_val[selected_features].to_dict(orient="records")
    X_val_new = dv_new.transform(val_dict_new)

    y_pred_new = model_new.predict(X_val_new)
    accuracy_new = (y_pred_new == y_val).mean()

    feature_results.append(
        (feature, float(accuracy_new), float(abs(baseline_accuracy - accuracy_new)))
    )

In [68]:
feature_results

[('lead_source', 0.642, 0.0030000000000000027),
 ('industry', 0.645, 0.0),
 ('employment_status', 0.645, 0.0),
 ('location', 0.645, 0.0),
 ('annual_income', 0.724, 0.07899999999999996),
 ('number_of_courses_viewed', 0.643, 0.0020000000000000018),
 ('interaction_count', 0.601, 0.04400000000000004),
 ('lead_score', 0.644, 0.0010000000000000009)]

In [74]:
ans = []

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    X_train = dv.transform(train_dict)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    val_dict = df_val[categorical_features + numerical_features].to_dict(orient="records")
    X_val = dv.transform(val_dict)
    
    y_pred = model.predict(X_val)

    ans.append((round(float((y_pred==y_val).mean()), 3), C))

In [75]:
ans.sort()

In [76]:
ans

[(0.598, 1e-06), (0.598, 1e-05), (0.613, 0.0001), (0.645, 0.001)]